In [1]:
# %%
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedShuffleSplit
from sklearn.metrics import make_scorer
import warnings
warnings.filterwarnings('ignore')

In [2]:
def compute_scatter_matrices(X, y):
    '''Computes the scatter matrices for a given dataset.'''

    classes = np.unique(y)
    n_features = X.shape[1]

    mu = np.mean(X, axis=0)

    W = np.zeros((n_features, n_features))
    B = np.zeros((n_features, n_features))

    for c in classes:
        Xc = X[y == c]

        mu_c = np.mean(Xc, axis=0)

        # Within class scatter
        diff = Xc - mu_c
        W += diff.T @ diff

        # Between class scatter
        n_c = Xc.shape[0]
        mean_diff = (mu_c - mu).reshape(-1, 1)
        B += n_c * (mean_diff @ mean_diff.T)

    W += np.eye(n_features) * 1e-6  # Regularization to ensure W is invertible

    return W, B

In [3]:
def compute_dann_transform(W, B, eps=1e-3):
    ''' Computes the DANN transformation matrix.'''

    eigvals, eigvecs = np.linalg.eigh(W)

    W_inv_sqrt = eigvecs @ np.diag(1/(np.sqrt(eigvals) + 1e-8)) @ eigvecs.T

    B_star = W_inv_sqrt @ B @ W_inv_sqrt

    sigma = W_inv_sqrt @ (B_star + eps*np.eye(W.shape[0])) @ W_inv_sqrt

    # Cholesky decomposition for L
    L = np.linalg.cholesky(sigma)

    return L

In [4]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.neighbors import NearestNeighbors


class LocalDANN(BaseEstimator, ClassifierMixin):

    def __init__(self, n_neighbors=5, k0=50, eps=1e-3):
        self.n_neighbors = n_neighbors
        self.k0 = k0
        self.eps = eps

    def fit(self, X, y):
        self.X = np.asarray(X)
        self.y = np.asarray(y)

        # Precompute Euclidean neighbor structure
        self.nbrs = NearestNeighbors(n_neighbors=self.k0, metric='euclidean')
        self.nbrs.fit(self.X)

        return self

    def _compute_scatter(self, Xn, yn):
        classes = np.unique(yn)
        d = Xn.shape[1]

        mu = np.mean(Xn, axis=0)

        W = np.zeros((d, d))
        B = np.zeros((d, d))

        for c in classes:
            Xc = Xn[yn == c]
            mu_c = np.mean(Xc, axis=0)

            diff = Xc - mu_c
            W += diff.T @ diff

            n_c = Xc.shape[0]
            md = (mu_c - mu).reshape(-1, 1)
            B += n_c * (md @ md.T)

        # regularization
        W += 1e-6 * np.eye(d)

        return W, B

    def _compute_metric(self, W, B):

        eigvals, eigvecs = np.linalg.eigh(W)
        W_inv_sqrt = eigvecs @ np.diag(1/np.sqrt(eigvals + 1e-10)) @ eigvecs.T

        B_star = W_inv_sqrt @ B @ W_inv_sqrt

        Sigma = W_inv_sqrt @ (B_star + self.eps * np.eye(W.shape[0])) @ W_inv_sqrt

        return Sigma

    def predict(self, Xq):

        Xq = np.asarray(Xq)
        preds = []

        for x in Xq:

            # Step 1: get k0 neighbors (Euclidean)
            _, idx = self.nbrs.kneighbors([x])
            # print("Got Neighbors")
            idx = idx[0]

            Xn = self.X[idx]
            yn = self.y[idx]

            # Step 2: local scatter
            W, B = self._compute_scatter(Xn, yn)
            # print("Computed Scatter")

            # Step 3: local metric
            Sigma = self._compute_metric(W, B)
            # print("Computed Sigma")
            # Step 4: compute distances (Mahalanobis)
            diff = self.X - x
            dists = np.sum((diff @ Sigma) * diff, axis=1)

            # Step 5: final KNN
            nn_idx = np.argsort(dists)[:self.n_neighbors]
            votes = self.y[nn_idx]

            pred = np.bincount(votes).argmax()
            preds.append(pred)
            # print("Completed prediction")

        return np.array(preds)

In [5]:
# %%
# ─────────────────────────────────────────────────────────────────────────────
# Theoretical Risk Functions  (Cover & Hart 1967 / Dist-KNN framework)
# ─────────────────────────────────────────────────────────────────────────────

def compute_r_star_x(model, X_query: pd.DataFrame) -> np.ndarray:
    """
    Conditional Bayes risk r*(x) at each query point x.

    For an M-class problem:
        r*(x) = 1 - max_c { q̂_c(x) }

    i.e. 1 minus the highest posterior class probability.
    This equals min(q̂₁, q̂₂) in the binary case.

    Parameters
    ----------
    model     : fitted GridSearchCV wrapping a KNeighborsClassifier (dictionary containing dist and prob model)
    X_query   : query points  (n_samples × n_features)

    Returns
    -------
    r_star : ndarray of shape (n_samples,)
    """
    prob_model = model['prob_model']
    q_hat = prob_model.predict_proba(X_query)          # (n_samples, n_classes)
    r_star = 1.0 - q_hat.max(axis=1)              # scalar per query point
    return r_star


def compute_R_star(model, X_query: pd.DataFrame) -> float:
    """
    Expected Bayes risk R* = E[r*(x)] for one machine,
    approximated by the sample mean over X_query.

    Parameters
    ----------
    model     : fitted GridSearchCV
    X_query   : held-out query points

    Returns
    -------
    R_star : float
    """
    return float(compute_r_star_x(model, X_query).mean())


def compute_mu_hat(model, X: pd.DataFrame) -> np.ndarray:
    """
    Proxy for μ(x): the model's predicted class probability vector at each point.
    Works for any number of classes — returns the full (n_samples, n_classes) matrix
    so that bias can be computed per class and then averaged into a scalar norm.

    Parameters
    ----------
    model : fitted GridSearchCV (dictionary for prob_model and dist_model)
    X     : feature matrix

    Returns
    -------
    mu_hat : ndarray of shape (n_samples, n_classes)
    """
    prob_model = model['prob_model']
    return prob_model.predict_proba(X)   # (n_samples, n_classes)


def compute_bias_vector(
    model,
    X_query: pd.DataFrame,
    K_i: int,
) -> np.ndarray:
    """
    Local bias for machine i, generalised to any number of classes:
        Bᵢ(x) = ‖μ(x) - (1/Kᵢ) Σⱼ μ(xᵢ,ⱼ)‖₂

    μ(x) is the full predicted class-probability vector (n_classes,).
    The per-class difference is averaged across classes via the L2 norm
    to produce a single scalar bias per query point, which is what the
    Dist-KNN covariance matrix C requires.

    Parameters
    ----------
    model   : fitted GridSearchCV (KNN inside)
    X_query : query points  (n_samples × n_features)
    K_i     : number of neighbours used by this machine

    Returns
    -------
    B_i : ndarray of shape (n_samples,)  — one scalar bias per query point
    """
    dist_model = model['dist_model']
    knn: KNeighborsClassifier = dist_model.best_estimator_

    # μ(x) — full probability vector at each query point: (n_samples, n_classes)
    mu_x = compute_mu_hat(model, X_query)

    # Retrieve the K_i nearest training-set neighbours for each query point
    distances, indices = knn.kneighbors(X_query, n_neighbors=K_i)

    # μ̂ at every training point: (n_train, n_classes)
    X_train_arr = knn._fit_X
    X_train_df  = pd.DataFrame(X_train_arr, columns=X_query.columns)
    mu_train    = compute_mu_hat(model, X_train_df)   # (n_train, n_classes)

    # Average μ over the K_i neighbours: indices is (n_samples, K_i)
    # mu_train[indices] → (n_samples, K_i, n_classes)
    mu_neighbour_means = mu_train[indices].mean(axis=1)  # (n_samples, n_classes)

    # Per-class difference, collapsed to a scalar via L2 norm — works for any n_classes
    B_i = np.linalg.norm(mu_x - mu_neighbour_means, axis=1)  # (n_samples,)
    return B_i


print("Theoretical risk functions defined.")

Theoretical risk functions defined.


In [6]:
def create_models(server_datasets):

    best_neighbors = []
    best_f1_scores = []
    model_instance_list = []
    dataset_sizes = []
        
    for df_train, df_test in (server_datasets):
        params1 = {"n_neighbors": np.arange(3, 51, 2)}
        knn = KNeighborsClassifier()
        model1 = GridSearchCV(knn, params1, scoring='f1_macro', cv=2, n_jobs=-1)
        # params2 = {"n_estimators": np.array([50, 100, 150, 200]), "max_depth": np.array([3, 5, 8, 10, 15])}
        # random_forests = RandomForestClassifier()
        # model2 = GridSearchCV(random_forests, params2, scoring='f1_macro', cv=2, n_jobs=-1)
        
        X_s_train = df_train.drop(['y'], axis=1).values
        y_s_train = df_train['y'].values
        X_s_test = df_test.drop(['y'], axis=1).values
        y_s_test = df_test['y'].values

        
        model1.fit(X_s_train, y_s_train)
        # model2.fit(X_s_train, y_s_train)
        y_pred = model1.predict(X_s_test)
        
        best_neighbors.append(model1.best_params_['n_neighbors'])
        best_f1_scores.append(f1_score(y_s_test, y_pred, average='macro'))
        model_instance_list.append({
            'dist_model': model1,
            'prob_model': model1
        })
        
        # # Compute median distances (possible logical flaw)
        # nbrs = model.best_estimator_
        # distances, indices = nbrs.kneighbors(X_s_train)
        # distances_no_self = distances[:, 1:]
        # median_neighbor_distance.append(np.median(distances_no_self))
        
        dataset_sizes.append(df_train.shape[0])

    return (best_neighbors, best_f1_scores, model_instance_list, dataset_sizes)

In [7]:
class ModelPerformanceAnalyzer:
    def __init__(self, models, model_names = None): # models : {dist_model: (knn model for neighbor), prob_model: (rf model for probabilities)}
        self.models = models
        self.model_names = model_names or [f"Model_{i}" for i in range(len(models))]
        self.performance_df = None
    
    def analyze_models(self, X_unseen, y_unseen):
        results = []
        for i, (model, name) in enumerate(zip(self.models, self.model_names)):
            
            model_metrics = self._analyze_single_model(
                model, name, i, X_unseen, y_unseen
            )
            results.append(model_metrics)
        
        self.performance_df = pd.DataFrame(results)
        return self.performance_df
    
    
    def _analyze_single_model(self, model, name, index, X_unseen, y_unseen):

        #Predictions and probabilities
        y_pred = model['prob_model'].predict(X_unseen)
        y_proba = model['prob_model'].predict_proba(X_unseen)

        #Distance metrics

        metrics = {
            'model_name': name,
            'model_index': index,
            'f1_score': f1_score(y_unseen, y_pred, average='macro'),
            'classification_probabilities': y_proba.tolist()
        }

        distance_metrics = self._calculate_distance_metrics(model['dist_model'], X_unseen)
        metrics.update(distance_metrics)
        return metrics


    def _calculate_distance_metrics(self, model, X_unseen):
        """Calculate various distance-based metrics."""
        try:
            nbrs = model.best_estimator_
            
            # Calculate distance matrix
            distances, indices = nbrs.kneighbors(X_unseen)
            distances_no_self = distances
            
            # Various distance statistics
            point_medians = np.median(distances_no_self, axis=1)
            point_means = np.mean(distances_no_self, axis=1)
            point_mins = np.min(distances_no_self, axis=1)
            point_maxs = np.max(distances_no_self, axis=1)
            
            return {
                'median_distance_all_points': np.median(point_medians),
                'mean_distance_all_points': np.mean(point_means),
                'min_distance_all_points': np.min(point_mins),
                'max_distance_all_points': np.max(point_maxs),
                'distance_std': np.std(distances_no_self),
            }
        except Exception as e:
            warnings.warn(f"Distance calculation failed for model: {e}")
            return {
                'median_distance_all_points': np.nan,
                'mean_distance_all_points': np.nan,
                'min_distance_all_points': np.nan,
                'max_distance_all_points': np.nan,
                'distance_std': np.nan,
            }
    def get_probability_matrix(self, model_index):
        """Get probability matrix for a specific model as numpy array."""
        if self.performance_df is not None:
            return np.array(self.performance_df.loc[model_index, 'classification_probabilities'])
        return None
    
    def get_all_probabilities(self):
        """Get all probability matrices as a 3D array (models x samples x classes)."""
        if self.performance_df is not None:
            return np.array([np.array(probs) for probs in self.performance_df['classification_probabilities']])
        return None

In [8]:
def run_distributed_knn_simulation(data_train, data_test, n_simulations=10, n_servers=100, test_size=0.2):
    """
    Optimized version using ModelPerformanceAnalyzer for probability extraction
    """
    import scipy.linalg as la

    simulation_results = []
    all_medians = []
    all_means = []

    for sim in range(n_simulations):
        print(f"Running simulation {sim+1}/{n_simulations}")

        # 1. Split data
        X_train_full   = data_train.drop(['y'], axis=1)
        y_train_full   = data_train['y']
        X_test_heldout = data_test.drop(['y'], axis=1)
        y_test_heldout = data_test['y']

        # 2. Generate server datasets
        def generate_server_datasets(train_pool, test_pool, n_servers=100,
                                     min_samples=1000, max_samples=5000):
            server_datasets = []
            X_train_pool    = train_pool.drop(columns=['y'])
            y_train_pool    = train_pool['y']
            X_test_pool     = test_pool.drop(columns=['y'])
            y_test_pool     = test_pool['y']

            for seed in range(n_servers):
                train_size = np.random.randint(min_samples, max_samples)
                sss_train  = StratifiedShuffleSplit(
                    n_splits=1, train_size=train_size, random_state=seed
                )
                train_idx, _ = next(sss_train.split(X_train_pool, y_train_pool))
                server_train  = train_pool.iloc[train_idx]

                test_size_s = int(0.3 * train_size)
                sss_test    = StratifiedShuffleSplit(
                    n_splits=1, train_size=test_size_s, random_state=seed
                )
                test_idx, _ = next(sss_test.split(X_test_pool, y_test_pool))
                server_test  = test_pool.iloc[test_idx]

                server_datasets.append((server_train, server_test))

            return server_datasets

        train_pool_df = pd.concat([X_train_full, y_train_full], axis=1)
        test_pool_df  = pd.concat([X_test_heldout, y_test_heldout], axis=1)

        server_datasets = generate_server_datasets(
            train_pool_df, test_pool_df, n_servers=n_servers
        )

        # 3. Train models
        best_neighbors, best_f1_scores, model_instance_list, dataset_sizes = create_models(server_datasets)

        metrics_df = pd.DataFrame({
            'optimal_k_value': best_neighbors,
            'best_f1_scores' : best_f1_scores,
            'model_instance' : model_instance_list,
            'dataset_size'   : dataset_sizes
        })
        metrics_df = metrics_df[metrics_df['best_f1_scores'] != 0].reset_index(drop=True)

        # 4. Test on held-out data
        n_test_samples = min(500, len(X_test_heldout))
        test_indices   = np.random.choice(len(X_test_heldout), size=n_test_samples, replace=False)
        X_test_samples = X_test_heldout.iloc[test_indices]
        y_test_true    = y_test_heldout.iloc[test_indices]

        # Use ModelPerformanceAnalyzer to get all metrics and probabilities
        analyzer    = ModelPerformanceAnalyzer(metrics_df['model_instance'].tolist())
        analysis_df = analyzer.analyze_models(X_test_samples, y_test_true)

        # Merge results
        metrics_df = metrics_df.merge(
            analysis_df[['model_index', 'median_distance_all_points', 'mean_distance_all_points']],
            left_index=True,
            right_on='model_index'
        ).reset_index(drop=True)

        all_medians.extend(metrics_df['median_distance_all_points'].tolist())
        all_means.extend(metrics_df['mean_distance_all_points'].tolist())

        # 5. Define scoring functions
        def arctan_score(distance, df):
            min_d = df['median_distance_all_points'].min()
            max_d = df['median_distance_all_points'].max()
            return 0.5 + (np.arctan(((max_d + min_d)/2) - distance) / np.pi)

        def tanh_score(distance, df):
            min_d = df['median_distance_all_points'].min()
            max_d = df['median_distance_all_points'].max()
            return 0.5 + 0.5 * (np.tanh(((max_d + min_d)/2) - distance))

        def sigmoid_score(distance, df):
            min_d = df['median_distance_all_points'].min()
            max_d = df['median_distance_all_points'].max()
            return 1 / (1 + 2 * np.exp(((max_d + min_d)/2) - distance))

        def relu_score(distance, df):
            mean      = df['mean_distance_all_points'].mean()
            deviation = df['mean_distance_all_points'].std()
            upper     = mean + 2 * deviation
            lower     = max(mean - 2 * deviation, 0)
            
            if distance > upper or distance < lower:
                return 0
            
            score = mean - abs(distance - mean)  # peaks at mean, falls off symmetrically
            return max(score, 0)

        metrics_df['arctan_score']  = metrics_df['median_distance_all_points'].apply(lambda x: arctan_score(x, metrics_df))
        metrics_df['tanh_score']    = metrics_df['median_distance_all_points'].apply(lambda x: tanh_score(x, metrics_df))
        metrics_df['sigmoid_score'] = metrics_df['median_distance_all_points'].apply(lambda x: sigmoid_score(x, metrics_df))
        metrics_df['relu_score']    = metrics_df['median_distance_all_points'].apply(lambda x: relu_score(x, metrics_df))

        # 6. Extract probabilities
        probability_list = []
        for i in range(len(X_test_samples)):
            point_probs = []
            for model_idx in range(len(metrics_df)):
                model_probs = analyzer.get_probability_matrix(model_idx)
                point_probs.append(model_probs[i])
            probability_list.append(point_probs)
        probability_list = np.array(probability_list)   # (n_query, M, n_classes)

        # 7. Theoretical risk quantities — computed BEFORE aggregation so W* can be used
        sim_models = metrics_df['model_instance'].tolist()
        sim_K_list = metrics_df['optimal_k_value'].tolist()
        M_sim      = len(sim_models)
        n_q        = len(X_test_samples)

        def _r_star_x(model, X):
            """r*(x) = 1 - max_c q̂_c(x)  — works for any number of classes."""
            return 1.0 - model['prob_model'].predict_proba(X).max(axis=1)

        def _bias_scalar_gwrr(model, X_query, K_i):
            """
            Vectorised GWRR spatial bias — gradient + Hessian via second-order polynomial
            design matrix with adaptive MISE-optimal bandwidth.

            All n_q query points are processed in a single batched solve:
              • One call to kneighbors for all queries
              • Design matrices built with broadcasting (no Python loops over queries)
              • Batched ridge solve via np.linalg.solve on stacked (n_q, p, p) system

            Bias formula (paper §3 / Theorem 2):
                bᵢ = gᵢᵀ dᵢ + ½ dᵢᵀ Hᵢ dᵢ
            """
            knn        = model['dist_model'].best_estimator_
            prob_model = model['prob_model']
            X_train    = knn._fit_X                              # (n_train, d)
            X_q_arr    = X_query.values                          # (n_q, d)
            n_q_local, d = X_q_arr.shape

            # Pre-compute class probabilities at every training point
            X_train_df = pd.DataFrame(X_train, columns=X_query.columns)
            p_train    = prob_model.predict_proba(X_train_df)   # (n_train, n_classes)

            # ── All neighbours at once ────────────────────────────────────────────────
            distances, indices = knn.kneighbors(X_query, n_neighbors=K_i)
            # distances: (n_q, K_i),  indices: (n_q, K_i)

            X_nbr   = X_train[indices]                           # (n_q, K_i, d)
            p_nbr   = p_train[indices]                           # (n_q, K_i, n_classes)

            # ── Step 1: centre neighbours around each query point ─────────────────────
            X_tilde = X_nbr - X_q_arr[:, np.newaxis, :]         # (n_q, K_i, d)

            # ── Step 2: adaptive MISE-optimal bandwidth per query ─────────────────────
            sigma_d = distances.std(axis=1) + 1e-8               # (n_q,)
            h_i     = sigma_d * (K_i ** (-1.0 / (d + 4)))        # (n_q,)

            # ── Step 3: Gaussian kernel weights ───────────────────────────────────────
            dist_sq = np.sum(X_tilde ** 2, axis=2)               # (n_q, K_i)
            omega   = np.exp(-dist_sq / (2.0 * h_i[:, np.newaxis] ** 2))  # (n_q, K_i)

            # ── Step 4: weighted centred target for dominant class ────────────────────
            mean_p      = p_nbr.mean(axis=1)                     # (n_q, n_classes)
            pred_class  = mean_p.argmax(axis=1)                  # (n_q,)
            # Binary indicator: 1 if neighbour's dominant class matches pred_class
            indicators  = (p_nbr.argmax(axis=2) ==
                           pred_class[:, np.newaxis]).astype(float)  # (n_q, K_i)
            omega_sum   = omega.sum(axis=1, keepdims=True) + 1e-12
            y_bar_w     = (omega * indicators).sum(axis=1, keepdims=True) / omega_sum
            y_tilde     = indicators - y_bar_w                    # (n_q, K_i)

            # ── Step 5: second-order polynomial design matrix ─────────────────────────
            # Linear: (n_q, K_i, d)
            Z_lin  = X_tilde

            # Pure quadratic: (n_q, K_i, d)
            Z_quad = X_tilde ** 2

            # Cross-interaction pairs
            cross_pairs = [(r, c) for r in range(d) for c in range(r + 1, d)]
            if cross_pairs:
                Z_cross = np.stack(
                    [X_tilde[:, :, r] * X_tilde[:, :, c] for r, c in cross_pairs],
                    axis=2
                )                                                 # (n_q, K_i, n_cross)
                Z_i = np.concatenate([Z_lin, Z_quad, Z_cross], axis=2)
            else:
                Z_i = np.concatenate([Z_lin, Z_quad], axis=2)    # (n_q, K_i, p)

            p_cols = Z_i.shape[2]
            alpha  = 1e-4

            # ── Step 6: batched GWRR solve ────────────────────────────────────────────
            # Scale each row of Z by sqrt(ω) → equivalent to weighted normal equations
            sqrt_w  = np.sqrt(omega)                              # (n_q, K_i)
            Zw      = Z_i * sqrt_w[:, :, np.newaxis]             # (n_q, K_i, p)
            yw      = y_tilde * sqrt_w                            # (n_q, K_i)

            # ZᵀZ + αI  and  Zᵀy — batched via einsum
            ZtWZ = np.einsum('nkp,nkq->npq', Zw, Zw)            # (n_q, p, p)
            ZtWZ += alpha * np.eye(p_cols)[np.newaxis]            # Tikhonov regularisation
            ZtWy = np.einsum('nkp,nk->np', Zw, yw)              # (n_q, p)

            # Solve all n_q systems at once
            theta = np.linalg.solve(ZtWZ, ZtWy)                  # (n_q, p)

            # ── Extract gradient ──────────────────────────────────────────────────────
            g_all = theta[:, :d]                                  # (n_q, d)

            # ── Extract Hessian ───────────────────────────────────────────────────────
            H_all = np.zeros((n_q_local, d, d))
            H_all[:, np.arange(d), np.arange(d)] = 2.0 * theta[:, d:2*d]
            for col_idx, (r, c) in enumerate(cross_pairs):
                H_all[:, r, c] = theta[:, 2*d + col_idx]
                H_all[:, c, r] = theta[:, 2*d + col_idx]

            # ── Step 7: coordinate-wise median displacement vector ────────────────────
            d_all = np.median(X_tilde, axis=1)                    # (n_q, d)

            # ── Step 8: second-order bias (vectorised) ────────────────────────────────
            b_linear  = np.einsum('nd,nd->n', g_all, d_all)      # gᵢᵀ dᵢ
            b_quad    = 0.5 * np.einsum('nd,nde,ne->n',
                                        d_all, H_all, d_all)      # ½ dᵢᵀ Hᵢ dᵢ
            b_scalars = b_linear + b_quad                         # (n_q,)

            return float(b_scalars.mean())
        # r*(x) and R* per server
        r_star_x_list, R_star_list = [], []
        for model in sim_models:
            r_x = _r_star_x(model, X_test_samples)
            r_star_x_list.append(r_x)
            R_star_list.append(float(r_x.mean()))

        r_star_matrix = np.stack(r_star_x_list, axis=0)   # (M, n_query)
        R_star_vec    = np.array(R_star_list)              # (M,)

        # bᵢ — paper-exact OLS scalar bias using median displacement (one scalar per server)
        # This is the quantity the paper says each client transmits (O(1) communication).
        b_vec = np.array([
            _bias_scalar_gwrr(model, X_test_samples, K_i)
            for model, K_i in zip(sim_models, sim_K_list)
        ])  # (M,)

        # C = b bᵀ — rank-1 outer product as defined in Theorem 2 of the paper
        C = np.outer(b_vec, b_vec)                         # (M, M)

        # V diagonal: Vᵢᵢ = R*ᵢ / Kᵢ
        V = np.diag(R_star_vec / np.array(sim_K_list, dtype=float))

        # Optimal W* via Sherman-Morrison (paper Section V, closed form)
        #   W* = (V+C)⁻¹ 1 / (1ᵀ (V+C)⁻¹ 1)
        # Pre-compute scalar sums for the Sherman-Morrison expansion
        K_vec  = np.array(sim_K_list, dtype=float)
        V_inv_diag = K_vec / R_star_vec                    # diagonal of V⁻¹
        S11    = np.sum(V_inv_diag)                        # 1ᵀ V⁻¹ 1
        Sb1    = float(b_vec @ V_inv_diag)                 # bᵀ V⁻¹ 1
        Sbb    = float(b_vec @ (V_inv_diag * b_vec))       # bᵀ V⁻¹ b

        denom_sm = S11 - (Sb1 ** 2) / (1.0 + Sbb)        # scalar denominator
        # Numerator: w*ᵢ ∝ (Kᵢ/R*ᵢ) · (1 − bᵢ · Sb1/(1+Sbb))
        num_vec  = V_inv_diag * (1.0 - b_vec * Sb1 / (1.0 + Sbb))
        W_star   = num_vec / denom_sm                      # (M,) — sums to 1 by construction

        # Clamp any numerically negative weights to zero and renormalise
        W_star   = np.clip(W_star, 0.0, None)
        W_star  /= W_star.sum()

        # Optimal R = R* + 1 / (1ᵀ (V+C)⁻¹ 1)  (Sherman-Morrison form)
        R_global_mean = float(R_star_vec.mean())
        R_optimal     = R_global_mean + 1.0 / denom_sm

        # Keep B_matrix for visualisation (store per-server bias scalar broadcast to n_query shape)
        B_matrix = b_vec[:, None] * np.ones((M_sim, n_q))  # (M, n_query) — constant per server

        # 8. Weighted predictions — approaches 1–5 (heuristic) + approach 6 (optimal W*)
        approaches = {
            'approach_1': 'arctan_score',
            'approach_2': 'dataset_size',
            'approach_3': 'tanh_score',
            'approach_4': 'sigmoid_score',
            'approach_5': 'relu_score'
        }

        approach_preds = {}
        for approach_name, weight_col in approaches.items():
            weights = metrics_df[weight_col].values
            weighted_results = []
            for probability in probability_list:
                weighted = probability * weights.reshape(-1, 1)
                weighted_results.append(weighted.sum(axis=0) / weights.sum())
            approach_preds[approach_name] = np.array([np.argmax(p) for p in weighted_results])

        # Approach 6 — optimal W* (already sums to 1, no normalisation needed)
        weighted_results_wstar = []
        for probability in probability_list:
            weighted = probability * W_star.reshape(-1, 1)   # (M, n_classes)
            weighted_results_wstar.append(weighted.sum(axis=0))
        approach_preds['approach_6'] = np.array([np.argmax(p) for p in weighted_results_wstar])

        # 9. Calculate metrics for all approaches
        sim_result = {'simulation': sim}
        for approach_name, preds in approach_preds.items():
            sim_result[f'{approach_name}_f1']  = f1_score(y_test_true, preds, average='macro')
            sim_result[f'{approach_name}_acc'] = accuracy_score(y_test_true, preds)

        sim_result.update({
            'n_servers'          : len(metrics_df),
            'avg_dataset_size'   : metrics_df['dataset_size'].mean(),
            'avg_median_distance': metrics_df['median_distance_all_points'].mean(),
            # Theoretical quantities
            'r_star_per_server'  : R_star_list,
            'r_star_x_matrix'    : r_star_matrix.tolist(),
            'bias_matrix'        : B_matrix.tolist(),
            'C_matrix'           : C.tolist(),
            'V_diag'             : np.diag(V).tolist(),
            'W_star'             : W_star.tolist(),
            'mean_R_star'        : R_global_mean,
            'min_R_star'         : float(R_star_vec.min()),
            'max_R_star'         : float(R_star_vec.max()),
            'optimal_R'          : R_optimal,
        })

        simulation_results.append(sim_result)

    results_df = pd.DataFrame(simulation_results)
    return results_df, all_medians, all_means

In [9]:
# %%
# Load your data
# data_train = pd.read_csv('/kaggle/input/datasets/abhirajraje/oelp-dataset/adult_preprocessed_train.csv')
# data_test = pd.read_csv('/kaggle/input/datasets/abhirajraje/oelp-dataset/adult_preprocessed_test.csv')

data_train = pd.read_csv('../data/letter_preprocessed_train.csv')
data_test = pd.read_csv('../data/letter_preprocessed_test.csv')

# Run multiple simulations
results_df, all_medians, all_means = run_distributed_knn_simulation(data_train, data_test, n_simulations=5, n_servers=10)

Running simulation 1/5


ValueError: solve: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (m,m),(m,n)->(m,n) (size 500 is different from 152)

In [ ]:
print(results_df)

In [ ]:
import scipy.stats as stats

plt.figure(figsize=(10, 5))
plt.hist(all_medians, bins=40)
plt.title('Distribution of Median Distances Across All Simulations')
plt.xlabel('Median Distance')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

plt.figure(figsize=(6, 6))
stats.probplot(all_medians, dist="norm", plot=plt)
plt.title('Q–Q Plot of Median Distances Across Simulations')
plt.show()

In [ ]:
import scipy.stats as stats

plt.figure(figsize=(10, 5))
plt.hist(all_means, bins=40)
plt.title('Distribution of Mean Distances Across All Simulations')
plt.xlabel('Mean Distance')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

plt.figure(figsize=(6, 6))
stats.probplot(all_means, dist="norm", plot=plt)
plt.title('Q–Q Plot of Mean Distances Across Simulations')
plt.show()

In [ ]:
# Find the best KNN on overall data
X_train = data_train.drop(['y'], axis=1)
y_train = data_train['y']
X_test = data_test.drop(['y'], axis=1)
y_test = data_test['y']


params1 = {"n_neighbors": np.arange(3, 31, 2)}
knn = KNeighborsClassifier()
model = GridSearchCV(knn, params1, scoring='f1_macro', cv=2, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

buffer_model = {"dist_model": model, "prob_model" : model} # Just there because compute_R_star accepts a dictionary

R_star_global_model = compute_R_star(buffer_model, X_test)
k_global_model = model.best_params_['n_neighbors']

V_global_model = R_star_global_model / k_global_model
print(classification_report(y_test, y_pred))
print(f"Best Params: {model.best_params_}")
print(f"Best F1 Score: {f1_score(y_test, y_pred, average='macro')}")

benchmark_f1 = f1_score(y_test, y_pred, average='macro')

In [ ]:
# %%
# Analyze results
print("Simulation Results Summary:")
print("=" * 50)
print(f"Number of simulations: {len(results_df)}")
print(f"Average F1 Scores:")
print(f"Approach 1 (Arctan): {results_df['approach_1_f1'].mean():.4f} ± {results_df['approach_1_f1'].std():.4f}")
print(f"Approach 2 (Size):   {results_df['approach_2_f1'].mean():.4f} ± {results_df['approach_2_f1'].std():.4f}")
print(f"Approach 3 (Tanh):   {results_df['approach_3_f1'].mean():.4f} ± {results_df['approach_3_f1'].std():.4f}")
print(f"Approach 4 (Sigmoid):   {results_df['approach_4_f1'].mean():.4f} ± {results_df['approach_4_f1'].std():.4f}")
print(f"Approach 5 (ReLU):   {results_df['approach_5_f1'].mean():.4f} ± {results_df['approach_5_f1'].std():.4f}")
print(f"Approach 6 (W*):   {results_df['approach_6_f1'].mean():.4f} ± {results_df['approach_6_f1'].std():.4f}")


In [ ]:
# %%
# Visualization
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
results_df[['approach_1_f1', 'approach_2_f1', 'approach_3_f1', 'approach_4_f1', 'approach_5_f1', 'approach_6_f1']].boxplot()
plt.title('F1 Score Distribution Across Simulations')
plt.ylabel('F1 Score')
plt.axhline(y=benchmark_f1, color='r', linestyle='--', label='Benchmark F1 Score')
plt.xticks([1, 2, 3, 4, 5, 6], ['Arctan', 'Size', 'Tanh', 'Sigmoid', 'RELU', 'W*'])

plt.subplot(1, 2, 2)
plt.plot(results_df['approach_1_f1'], label='Arctan', marker='o')
plt.plot(results_df['approach_2_f1'], label='Size', marker='s')
plt.plot(results_df['approach_3_f1'], label='Tanh', marker='^')
plt.plot(results_df['approach_4_f1'], label='Sigmoid', marker='X')
plt.plot(results_df['approach_5_f1'], label='RELU', marker='*')
plt.plot(results_df['approach_6_f1'], label='W*', marker='p')
plt.axhline(y=benchmark_f1, color='r', linestyle='--', label='Benchmark F1 Score')
plt.xticks(np.arange(len(results_df)), np.arange(1, len(results_df)+1))
plt.xlabel('Simulation')
plt.ylabel('F1 Score')
plt.title('F1 Score Trend Across Simulations')
plt.legend()
plt.grid(True)

# plt.savefig('f1_score_trends_std_normal.png')

plt.tight_layout()
plt.show()


## Theoretical Risk Analysis (Cover & Hart, 1967 + Dist-KNN)

The following cells compute the theoretical quantities from the Dist-KNN paper:

- **r\*(x)**: Conditional Bayes risk at point x — `min(q̂₁(x), q̂₂(x))` for binary classification  
- **R\***: Expected Bayes risk for each machine — `E[r*(x)]`  
- **Bᵢ(x)**: Local bias for machine i — `gᵢᵀdᵢ + ½dᵢᵀHᵢdᵢ` via GWRR with adaptive bandwidth and second-order polynomial design matrix  
- **C**: Uncentered covariance matrix where `C[i,j] = E[Bᵢ(x) Bⱼ(x)]`  
- **R**: Optimal aggregated risk under MSE — `R* + 1 / (1ᵀ (V+C)⁻¹ 1)`